\newpage

# Posterior distributions of calibrated parameters (base case analysis)

In [1]:
from pathlib import Path
from math import ceil
import itertools
import yaml

import numpy as np
import arviz as az
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from IPython.display import display, Markdown

import tbh.plotting as pl
import tbh.runner_tools as rt
from tbh.model import get_tb_model
from tbh.paths import REPO_ROOT_PATH
from estival.model import BayesianCompartmentalModel

NOTEBOOK_DIR = REPO_ROOT_PATH / "notebooks"
BASE_DIR = REPO_ROOT_PATH / "remote_cluster" / "outputs" / "59293003_longer_runs"
SA_BASE_DIR = REPO_ROOT_PATH / "remote_cluster" / "outputs" / "59223189_sas"

FIG_FILES = [
    ("figure_1_calibration.pdf", "Figure 1. Calibration outputs"),
    ("figure_2_trajectories.pdf", "Figure 2. Projected trajectories"),
    ("figure_4_algorithms_coverage.pdf", "Figure 4. Algorithm and coverage comparison"),
]

In [2]:
basecase_task_path = BASE_DIR / "task_6"
idata = az.from_netcdf(basecase_task_path / "idata.nc")
with open(basecase_task_path / "details.yaml", "r") as f:
    docs = list(yaml.safe_load_all(f))
model_config = docs[1] if len(docs) > 1 else {}
analysis_config = docs[2] if len(docs) > 2 else {}
params, priors, tv_params = rt.get_parameters_and_priors()

model = get_tb_model(model_config, tv_params)
bcm = BayesianCompartmentalModel(model, params, priors, rt.targets)

display(Markdown("---"))
display(Markdown(f"## Posterior vs prior"))
# display(Markdown(f"Config: `{cfg_label}`"))

fig = pl.plot_post_prior_comparison(
    idata,
    analysis_config['burn_in'],
    req_vars=list(bcm.priors.keys()),
    priors=list(bcm.priors.values()),
)

# fig.suptitle(f"{task_name} | {cfg_label}", y=1.02, fontsize=12)
display(fig)
plt.close(fig)

---

## Posterior vs prior

<Figure size 7590x6037.5 with 20 Axes>

In [4]:
burn_in = analysis_config['burn_in']
display(Markdown(f"## Parameter traces by chain"))

fig = pl.plot_traces(idata, bcm, burn_in=burn_in)

fig.tight_layout()
display(fig)
plt.close(fig)

## Parameter traces by chain

<Figure size 5850x5880 with 19 Axes>

In [5]:
import pandas as pd
from pathlib import Path

params_xlsx = Path("../data/parameters.xlsx")
df = pd.read_excel(params_xlsx, sheet_name="constant")

# The posterior figure plots all calibration priors: req_vars = list(bcm.priors.keys())
posterior_params = df.loc[df["distribution"].notna(), ["parameter", "full_text"]].copy()
posterior_params = posterior_params.drop_duplicates(subset=["parameter"]).sort_values("parameter")
posterior_params = posterior_params.rename(columns={"parameter": "parameter key", "full_text": "definition"})

In [6]:
#| echo: false
#| output: asis

latex_table = posterior_params.to_latex(
    index=False,
    escape=True,
    column_format=r"p{0.29\linewidth} p{0.66\linewidth}"
)
print(r"\begingroup")
print(r"\renewcommand{\arraystretch}{1.35}")
print(latex_table)
print(r"\endgroup")

\begingroup
\renewcommand{\arraystretch}{1.35}
\begin{tabular}{p{0.29\linewidth} p{0.66\linewidth}}
\toprule
parameter key & definition \\
\midrule
a\_spread & Spread of assortative mixing pattern (smaller value means more assortativity) \\
bg\_mixing & Background age-agnostic mixing level \\
breakdown\_rate & Rate of transition from 'contained' to 'incipient' (all ages) \\
clearance\_rate & Rate of transition from 'contained' to 'cleared' (all ages) \\
clinical\_progression\_rate & Rate of progression from subclinical to clinical TB \\
infection\_pop\_scale & Exponent for population scaling in force of infection calculation \\
infectiousness\_gain\_rate & Rate of progression from 'less infectious' to 'more infectious' TB \\
passive\_detection\_inflection & Time when passive detection started to scale up \\
passive\_detection\_past\_frac & Past passive detection rate, as a fraction of the current one \\
pc\_strength & Strength of parent-children mixing pattern \\
prev\_se\_cleared\_tst

# Posterior distributions of TB-spectrum progression rates for different regression rates

# Detailed sensitivity analyses

In [8]:
#| output: asis
# PDF layout: Figure 1 full width, then Figure 2 and Figure 4 side-by-side using LaTeX minipages.
def load_task_to_config_map(output_root: Path):
    config_map_path = output_root / "task_config_map.yaml"
    if not config_map_path.exists():
        return {}

    with open(config_map_path, "r") as f:
        raw = yaml.safe_load(f)

    if not isinstance(raw, dict):
        return {}

    source = raw.get("tasks", raw)
    if not isinstance(source, dict):
        return {}

    task_to_config = source.get("task_to_config", {})
    return task_to_config if isinstance(task_to_config, dict) else {}


def get_task_config_from_map(task_name: str, task_to_config: dict):
    try:
        task_id = int(task_name.split("_")[1])
    except Exception:
        return {}

    return task_to_config.get(task_id, task_to_config.get(str(task_id), {}))


def load_details_docs(task_path: Path):
    details_path = task_path / "details.yaml"
    if not details_path.exists():
        return []
    with open(details_path, "r") as f:
        return list(yaml.safe_load_all(f))


def get_model_config_from_details(docs):
    if len(docs) > 1 and isinstance(docs[1], dict):
        return docs[1]
    return {}


def get_analysis_config_from_details(docs):
    if len(docs) > 2 and isinstance(docs[2], dict):
        return docs[2]
    return {}


import os

def rel_to_notebook(path: Path):
    return os.path.relpath(path, NOTEBOOK_DIR)


def task_has_any_figure(task_path: Path):
    return any((task_path / fname).exists() for fname, _ in FIG_FILES)


task_roots = [
    ("Base outputs", BASE_DIR),
    ("Sensitivity analyses", SA_BASE_DIR),
]

lines = []

for root_label, root_dir in task_roots:
    task_to_config = load_task_to_config_map(root_dir)
    task_dirs = sorted([
        p for p in root_dir.iterdir()
        if p.is_dir() and (p / "details.yaml").exists() and task_has_any_figure(p)
    ])

    for task_path in task_dirs:
        task_name = task_path.name
        map_cfg = get_task_config_from_map(task_name, task_to_config)
        docs = load_details_docs(task_path)
        model_cfg = get_model_config_from_details(docs)
        analysis_cfg = get_analysis_config_from_details(docs)

        rel_sus = map_cfg.get("rel_sus_unreachable", "NA") if isinstance(map_cfg, dict) else "NA"
        reg = map_cfg.get("clinical_regression_rate", "NA") if isinstance(map_cfg, dict) else "NA"

        iso3 = model_cfg.get("iso3", "NA") if isinstance(model_cfg, dict) else "NA"
        hetero = model_cfg.get("heterogeneous_mixing", "NA") if isinstance(model_cfg, dict) else "NA"

        sensitivity_name = "NA"
        if isinstance(analysis_cfg, dict):
            sensitivity_name = analysis_cfg.get("sensitivity_analysis", "NA")

        lines.append(f"## {root_label} - {task_name}")
        lines.append("")
        lines.append("Minimal configuration summary:")
        lines.append(f"- Output root: {root_dir.name}")
        lines.append(f"- Task folder: {task_name}")
        lines.append(f"- ISO3: {iso3}")
        lines.append(f"- Heterogeneous mixing: {hetero}")
        lines.append(f"- Relative susceptibility (unreachable): {rel_sus}")
        lines.append(f"- Clinical regression rate: {reg}")
        lines.append(f"- Sensitivity analysis tag: {sensitivity_name}")
        lines.append("")

        fig1_path = task_path / "figure_1_calibration.pdf"
        fig2_path = task_path / "figure_2_trajectories.pdf"
        fig4_path = task_path / "figure_4_algorithms_coverage.pdf"

        if fig1_path.exists():
            fig1_rel = rel_to_notebook(fig1_path)
            lines.append("\\begin{center}")
            lines.append(f"\\includegraphics[width=1.00\\textwidth]{{{fig1_rel}}}")
            lines.append("\\end{center}")
        else:
            lines.append("Not available.")
        lines.append("")

        lines.append("\\begin{minipage}[t]{0.49\\textwidth}")
        if fig2_path.exists():
            fig2_rel = rel_to_notebook(fig2_path)
            lines.append(f"\\includegraphics[width=\\linewidth]{{{fig2_rel}}}")
        else:
            lines.append("\\textit{Not available.}")
        lines.append("\\end{minipage}\\hfill")

        lines.append("\\begin{minipage}[t]{0.49\\textwidth}")
        if fig4_path.exists():
            fig4_rel = rel_to_notebook(fig4_path)
            lines.append(f"\\includegraphics[width=\\linewidth]{{{fig4_rel}}}")
        else:
            lines.append("\\textit{Not available.}")
        lines.append("\\end{minipage}")
        lines.append("")

        lines.append("\\newpage")
        lines.append("")

print("\n".join(lines))

## Base outputs - task_1

Minimal configuration summary:
- Output root: 59293003_longer_runs
- Task folder: task_1
- ISO3: KIR
- Heterogeneous mixing: True
- Relative susceptibility (unreachable): 1.0
- Clinical regression rate: 0.5
- Sensitivity analysis tag: NA

\begin{center}
\includegraphics[width=1.00\textwidth]{../remote_cluster/outputs/59293003_longer_runs/task_1/figure_1_calibration.pdf}
\end{center}

\begin{minipage}[t]{0.49\textwidth}
\includegraphics[width=\linewidth]{../remote_cluster/outputs/59293003_longer_runs/task_1/figure_2_trajectories.pdf}
\end{minipage}\hfill
\begin{minipage}[t]{0.49\textwidth}
\includegraphics[width=\linewidth]{../remote_cluster/outputs/59293003_longer_runs/task_1/figure_4_algorithms_coverage.pdf}
\end{minipage}

\newpage

## Base outputs - task_10

Minimal configuration summary:
- Output root: 59293003_longer_runs
- Task folder: task_10
- ISO3: KIR
- Heterogeneous mixing: True
- Relative susceptibility (unreachable): 1.5
- Clinical regression rate